In [1]:
# Install required libraries first: pip install transformers datasets evaluate scikit-learn gradio
import evaluate
import numpy as np
import gradio as gr
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

# 1. Dataset Loading & Preprocessing
dataset = load_dataset("ag_news")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# For quick testing, we use a small subset of the dataset
train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(1000))
eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(200))

# 2. Model Development & Training
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=4)

metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

# Train the model
trainer.train()

# 3. Evaluation
eval_results = trainer.evaluate()
print(f"Evaluation Results: {eval_results}")

# 4. Deployment using Gradio (Live Interaction)
def classify_news(text):
    inputs = tokenizer(text, return_tensors="pt")
    outputs = model(**inputs)
    predictions = np.argmax(outputs.logits.detach().numpy(), axis=-1)
    labels = ["World", "Sports", "Business", "Sci/Tech"]
    return labels[predictions[0]]

interface = gr.Interface(fn=classify_news, inputs="text", outputs="text", title="News Topic Classifier")
# Uncomment the line below to launch the web app
# interface.launch()

ModuleNotFoundError: No module named 'evaluate'

In [2]:
# Install required libraries first: pip install pandas scikit-learn joblib
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# 1. Dataset Loading (Mocking Telco Churn Data for demonstration)
data = {
    'tenure': [12, 45, 2, 50, 1, 60],
    'MonthlyCharges': [50.5, 89.0, 20.0, 105.0, 30.0, 110.0],
    'Contract': ['Month-to-month', 'One year', 'Month-to-month', 'Two year', 'Month-to-month', 'Two year'],
    'InternetService': ['DSL', 'Fiber optic', 'No', 'Fiber optic', 'DSL', 'Fiber optic'],
    'Churn': [1, 0, 1, 0, 1, 0] # 1: Yes, 0: No
}
df = pd.DataFrame(data)

X = df.drop('Churn', axis=1)
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Preprocessing Steps
numeric_features = ['tenure', 'MonthlyCharges']
numeric_transformer = Pipeline(steps=[('scaler', StandardScaler())])

categorical_features = ['Contract', 'InternetService']
categorical_transformer = Pipeline(steps=[('onehot', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# 3. Model Pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# 4. Hyperparameter Tuning using GridSearchCV
param_grid = {
    'classifier__n_estimators': [10, 50, 100],
    'classifier__max_depth': [None, 5, 10]
}

grid_search = GridSearchCV(pipeline, param_grid, cv=2, scoring='accuracy')
grid_search.fit(X_train, y_train)

print(f"Best Parameters: {grid_search.best_params_}")

# 5. Evaluation
y_pred = grid_search.predict(X_test)
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# 6. Exporting the complete pipeline
joblib.dump(grid_search.best_estimator_, 'churn_pipeline_model.pkl')
print("Pipeline saved successfully as 'churn_pipeline_model.pkl'")

Best Parameters: {'classifier__max_depth': None, 'classifier__n_estimators': 10}

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         1

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2

Pipeline saved successfully as 'churn_pipeline_model.pkl'


In [ ]:
# Install required libraries first: pip install transformers pandas
import pandas as pd
from transformers import pipeline

# 1. Dataset Loading (Mock Free-text Support Ticket Dataset)
tickets = [
    "I cannot access my account, it keeps saying password incorrect even after I reset it.",
    "The website is very slow today and images are not loading on the checkout page.",
    "Can you please send me the invoice for my last month's subscription?",
    "How do I upgrade my current plan to the premium tier?"
]

# Define candidate tags for categorization
candidate_tags = ["Authentication", "Billing", "Technical Issue", "Account Upgrade", "Bug Report"]

# 2. Load Zero-Shot Classification LLM Pipeline
# We use a lightweight model suitable for zero-shot text classification
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# 3. Prediction & Outputting Top 3 Tags
results = []
for ticket in tickets:
    # Get predictions
    prediction = classifier(ticket, candidate_tags, multi_label=False)
    
    # Extract top 3 tags and their scores
    top_3_tags = prediction['labels'][:3]
    top_3_scores = prediction['scores'][:3]
    
    results.append({
        "Ticket": ticket,
        "Top_1_Tag": top_3_tags[0],
        "Top_2_Tag": top_3_tags[1],
        "Top_3_Tag": top_3_tags[2]
    })

# Convert to DataFrame for a clean view
df_results = pd.DataFrame(results)

# 4. Final Summary / Insights
print("Auto-Tagging Results (Top 3 Tags per Ticket):\n")
for index, row in df_results.iterrows():
    print(f"Ticket: {row['Ticket']}")
    print(f"Tags: 1. {row['Top_1_Tag']} | 2. {row['Top_2_Tag']} | 3. {row['Top_3_Tag']}\n")

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

C:\Users\PMLS\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\PMLS\.cache\huggingface\hub\models--facebook--bart-large-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\PMLS\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:744: UserWarning: Not enough free disk space t

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

C:\Users\PMLS\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:744: UserWarning: Not enough free disk space to download the file. The expected file size is: 1629.44 MB. The target location C:\Users\PMLS\.cache\huggingface\hub\models--facebook--bart-large-mnli\blobs only has 184.62 MB free disk space.
  warnings.warn(


model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

In [ ]:
!pip install evaluate transformers datasets scikit-learn gradio